<a href="https://colab.research.google.com/github/luiscano2330-bot/curso-programacion-cientifica/blob/main/Entregable_1_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto integrador — Grupo 2: Incidentes de seguridad en Medellín (2025)

**Entregable 1 — Análisis y limpieza de datos con pandas**

- **Integrantes:** Luis Steven Cano Gomez
- **Grupo:** 2
- **Repositorio:** `proyecto-integrador-pc-grupo-2`



## 1. Carga e inspección del conjunto de datos

Antes de analizar nada necesito saber qué tengo: cuántas filas y columnas hay, qué tipo tiene cada columna y qué valores aparecen. El archivo se lee con una **ruta relativa** (`data/...`) porque así el notebook funciona igual en mi computador y en Colab cuando el CSV está en la carpeta `data/` del repositorio.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

ruta_csv = "data/seguridad_incidentes_medellin_2025.csv"

# En Colab el CSV se sube al panel de archivos; si quedó en la raíz, lo paso a data/
if not Path(ruta_csv).exists():
    Path("data").mkdir(exist_ok=True)
    if Path("seguridad_incidentes_medellin_2025.csv").exists():
        Path("seguridad_incidentes_medellin_2025.csv").replace(ruta_csv)
    else:
        from google.colab import files
        print("Sube el archivo seguridad_incidentes_medellin_2025.csv")
        subidos = files.upload()
        Path(next(iter(subidos))).replace(ruta_csv)

df = pd.read_csv(ruta_csv)
print("El archivo tiene", df.shape[0], "filas y", df.shape[1], "columnas")

Sube el archivo seguridad_incidentes_medellin_2025.csv


El archivo se cargó completo: **4.505 filas y 19 columnas**. Cada fila es un incidente reportado en Medellín durante 2025 y cada columna es una característica de ese incidente (fecha, comuna, tipo, arma, víctima, etc.). No hubo errores de lectura, así que el separador y la codificación del archivo son los esperados.

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df.dtypes.to_frame("tipo_de_dato")

`head()` muestra que las columnas de texto traen tildes y espacios tal como las escribió quien reportó el incidente, y `info()` confirma que **todas las columnas tienen 4.505 datos, salvo seis que tienen menos**: `iluminacion_zona`, `edad_victima`, `bien_hurtado`, `valor_perdido_cop`, `tiempo_respuesta_min` y `testigos`. Esas ausencias son el primer problema de calidad que voy a revisar.

En los tipos aparece algo raro: **`fecha` está guardada como texto (`str`)** y no como fecha. Mientras sea texto no puedo ordenar por fecha ni sacar el mes con `.dt`, así que la corrijo más abajo. `hora` y `mes` sí son enteros, y las demás columnas numéricas son decimales.

In [ ]:
df.describe().round(2)

`describe()` resume las columnas numéricas. Lo primero que salta a la vista es `valor_perdido_cop`: la media (≈ 2,79 millones) es casi el doble de la mediana (≈ 1,45 millones) y el máximo llega a **120 millones de pesos**. Esa diferencia entre media y mediana avisa de valores extremos que voy a revisar.

`edad_victima` va de 12 a 72 años, un rango creíble. `hora` va de 0 a 23 y `mes` de 1 a 12, así que no hay horas ni meses imposibles. `tiempo_respuesta_min` va de 3 a 122 minutos y `testigos` de 0 a 6.

In [ ]:
print("Tipos de incidente:")
print(df["tipo_incidente"].value_counts())
print()
print("Lugares:")
print(df["lugar"].value_counts())
print()
print("Armas:")
print(df["arma"].value_counts())
print()
print("Comunas distintas escritas:", df["comuna"].nunique())
print(df["comuna"].value_counts().head(20))

Los **hurtos** son el grupo más grande: hurto a persona (1.367) y hurto de celular (1.252) suman más de la mitad de los casos, y a ellos se suma el hurto de moto (401). Los delitos de convivencia (riña, violencia intrafamiliar y lesiones) son menos frecuentes.

La mayoría de los incidentes ocurre en **vía pública** (2.457 de 4.505), lo que tiene sentido para hurtos y riñas.

En `comuna` aparece el primer problema serio: **52 formas distintas de escribir solo 16 comunas**. "La Candelaria", "LA CANDELARIA", "la candelaria" y " La Candelaria" son la misma comuna, pero pandas las cuenta como grupos separados. Si agrupo sin corregir, los promedios de esas comunas salen con muy pocos datos y no sirven.

In [ ]:
print("Antes:", df["fecha"].dtype)
df["fecha"] = pd.to_datetime(df["fecha"])
print("Después:", df["fecha"].dtype)
print("Primer día:", df["fecha"].min().date(), "| Último día:", df["fecha"].max().date())
print("Días distintos con reportes:", df["fecha"].nunique())

Convierto `fecha` a tipo fecha porque el diccionario dice que es una fecha y porque así puedo usar `.dt` para sacar el mes o el día de la semana. La conversión funcionó sin errores (no hubo fechas mal escritas) y cubre todo 2025: del **1 de enero al 31 de diciembre**, con 365 días distintos. Ese es el periodo que voy a analizar.

## 2. Diagnóstico de calidad de los datos

Ya sé qué hay. Ahora reviso **qué está mal**: faltantes, faltantes que en realidad significan "no aplica", valores imposibles, códigos raros, categorías inconsistentes y filas repetidas. La idea es tener la evidencia antes de decidir qué hacer con cada problema.

In [ ]:
tabla_faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(1),
})
tabla_faltantes.sort_values("faltantes", ascending=False).head(10)

Seis columnas tienen faltantes:

- `testigos`: **3.224 de 4.505 (71,6 %)**. Es demasiado: con casi tres de cada cuatro datos vacíos no se puede concluir nada serio.
- `bien_hurtado` y `valor_perdido_cop`: **1.485 cada una (33,0 %)**. El mismo número en las dos columnas sugiere que faltan juntas, y eso hay que verificarlo.
- `tiempo_respuesta_min`: 550 (12,2 %).
- `edad_victima`: 393 (8,7 %).
- `iluminacion_zona`: 219 (4,9 %).

No todos esos vacíos significan lo mismo, así que los reviso por separado antes de rellenar nada.

In [ ]:
pd.crosstab(df["tipo_incidente"], df["bien_hurtado"].isna(), margins=True)

Aquí está la clave: `bien_hurtado` y `valor_perdido_cop` **solo faltan en riñas, lesiones personales y violencia intrafamiliar**, y no falta ni una en los hurtos. Es decir, no son datos perdidos: son **"no aplica"**. En una riña no hay un bien hurtado ni un valor perdido que registrar.

Eso cambia la decisión: **estos vacíos no se imputan**. Rellenarlos con la mediana inventaría un robo que nunca ocurrió y además mezclaría delitos de hurto con delitos de convivencia.

In [ ]:
def contar_atipicos(serie):
    """Cuenta los valores fuera del rango Q1 - 1.5*IQR a Q3 + 1.5*IQR."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    fuera = (serie < limite_inferior) | (serie > limite_superior)
    return int(fuera.sum()), round(limite_inferior, 2), round(limite_superior, 2)

for columna in ["edad_victima", "valor_perdido_cop", "tiempo_respuesta_min"]:
    cantidad, limite_inferior, limite_superior = contar_atipicos(df[columna])
    print(f"{columna}: {cantidad} atípicos | rango normal [{limite_inferior} ; {limite_superior}]")

In [ ]:
df.loc[df["valor_perdido_cop"] > 50_000_000,
       ["id_incidente", "comuna", "tipo_incidente", "bien_hurtado", "valor_perdido_cop"]
      ].sort_values("valor_perdido_cop", ascending=False)

El criterio IQR marca 13 edades, 331 valores perdidos y 153 tiempos de respuesta. Pero **"atípico" no es lo mismo que "error"**, así que miro cada caso:

- **Edad (13 casos):** son personas de más de 61 años. Un adulto mayor víctima de un delito es perfectamente posible, así que se conservan.
- **Tiempo de respuesta (153 casos):** son demoras de más de 56 minutos. Son demoras reales, no errores de digitación. Se conservan.
- **Valor perdido (331 casos):** la mayoría son motos y computadores caros, que son plausibles. Pero al filtrar los valores por encima de 50 millones aparecen **5 casos imposibles**: cuatro registros de **120.000.000 COP** (dos celulares y dos bolsos) y una moto de 52.589.000 COP. Un celular no vale 120 millones; eso es un error de digitación (probablemente ceros de más). Esos 5 sí se corrigen.

Esto es importante para la exposición: el criterio automático solo **señala**; la decisión de borrar o conservar la toma quien conoce el dato.

In [ ]:
print("Formas de escribir las comunas:", df["comuna"].nunique())
print(df["comuna"].value_counts().tail(12))
print()
print("Valores de la columna arma:")
print(df["arma"].value_counts())

Además de las comunas repetidas con mayúsculas, minúsculas y espacios, aparece un código raro: **`arma = "Sin dato"` en 314 casos (7,0 %)**. No es un vacío técnico, es una categoría que significa "no se logró identificar el arma". La dejo como categoría propia y no la imputo, porque cambiarla por "Sin arma" afirmaría algo que no sé.

No hay otras categorías mal escritas en `tipo_incidente`, `modalidad`, `lugar` ni `sexo_victima`.

In [ ]:
print("Filas completamente repetidas:", df.duplicated().sum())
print("Identificadores repetidos:", df["id_incidente"].duplicated().sum())
df.loc[df.duplicated(keep=False), ["id_incidente", "fecha", "comuna", "tipo_incidente"]].sort_values("id_incidente").head(8)

Hay **25 filas completamente repetidas** (el mismo incidente con el mismo `id_incidente` y todos los mismos valores). No son dos incidentes distintos: son el mismo registro duplicado, probablemente por una doble carga del archivo. Si no las quito, los conteos por comuna y por tipo quedan inflados y los promedios se calculan dos veces sobre el mismo caso.

## 3. Limpieza de los datos

Con el diagnóstico hecho, aplico las decisiones una por una y **anoto el costo de cada una**: cuántas filas o valores cambian. Trabajo sobre una copia (`df_limpio`) para no dañar los datos originales y poder comparar.

El plan es:

1. Eliminar las 25 filas duplicadas.
2. Unificar las 52 formas de escribir las comunas.
3. Eliminar la columna `testigos` (71,6 % de faltantes).
4. Corregir los 5 valores imposibles de `valor_perdido_cop`.
5. Imputar `iluminacion_zona` con la **moda**.
6. Imputar `edad_victima` con la **mediana global**.
7. Imputar `tiempo_respuesta_min` con la **mediana por comuna**.
8. No tocar los faltantes que significan "no aplica".

In [ ]:
filas_antes = len(df)

df_limpio = df.drop_duplicates().reset_index(drop=True)

print("Filas antes:", filas_antes)
print("Filas después:", len(df_limpio))
print("Filas eliminadas:", filas_antes - len(df_limpio))
print("Identificadores repetidos que quedan:", df_limpio["id_incidente"].duplicated().sum())

**Costo:** se pierden **25 filas** (0,6 % del total) y cada `id_incidente` queda una sola vez. Es un costo bajo y evita contar dos veces los mismos incidentes.

In [ ]:
import unicodedata

def normalizar_texto(texto):
    """Pasa un texto a minúsculas, sin espacios externos y sin tildes."""
    limpio = str(texto).strip().lower()
    sin_tildes = "".join(
        caracter for caracter in unicodedata.normalize("NFKD", limpio)
        if not unicodedata.combining(caracter)
    )
    return sin_tildes

mapa_comunas = {
    "la candelaria": "La Candelaria", "el poblado": "El Poblado",
    "laureles estadio": "Laureles Estadio", "belen": "Belén",
    "robledo": "Robledo", "buenos aires": "Buenos Aires",
    "la america": "La América", "aranjuez": "Aranjuez",
    "castilla": "Castilla", "manrique": "Manrique",
    "san javier": "San Javier", "popular": "Popular",
    "guayabal": "Guayabal", "doce de octubre": "Doce de Octubre",
    "villa hermosa": "Villa Hermosa", "santa cruz": "Santa Cruz",
}

df_limpio["comuna_original"] = df_limpio["comuna"]
df_limpio["comuna"] = df_limpio["comuna"].map(normalizar_texto).map(mapa_comunas)
comunas_corregidas = int((df_limpio["comuna_original"] != df_limpio["comuna"]).sum())
df_limpio = df_limpio.drop(columns="comuna_original")

print("Comunas distintas antes: 52")
print("Comunas distintas después:", df_limpio["comuna"].nunique())
print("Filas cuya comuna se corrigió:", comunas_corregidas)
print("Comunas sin nombre reconocido:", df_limpio["comuna"].isna().sum())
print()
print(df_limpio["comuna"].value_counts())

**Costo:** se reescriben **77 filas** (las que estaban en mayúsculas, minúsculas o con espacios), pero **no se elimina ninguna**. Las 52 categorías quedan en **16 comunas** y ninguna fila se queda sin comuna. Ahora sí puedo agrupar por comuna sin que un promedio salga calculado con 1 o 2 casos mal escritos.

In [ ]:
columnas_antes = df_limpio.shape[1]
faltantes_testigos = int(df_limpio["testigos"].isna().sum())
porcentaje_testigos = df_limpio["testigos"].isna().mean() * 100

df_limpio = df_limpio.drop(columns="testigos")

print("Columnas antes:", columnas_antes, "| Columnas después:", df_limpio.shape[1])
print(f"testigos tenía {faltantes_testigos} faltantes ({porcentaje_testigos:.1f} %)")
print("Filas: no cambian ->", len(df_limpio))

**Decisión:** elimino `testigos` porque **7 de cada 10 celdas están vacías** (3.204 de 4.480, el 71,5 %). Imputarla con la moda (que es 0) diría que en casi todos los incidentes no hubo testigos, cuando en realidad no sabemos si los hubo: es un dato faltante, no un cero. Conservar la columna con ese nivel de vacío solo aportaría ruido.

**Costo:** se pierde una columna, pero **no se pierde ninguna fila**; las otras 18 columnas siguen completas.

In [ ]:
imposibles = df_limpio["valor_perdido_cop"] > 50_000_000
print("Valores imposibles encontrados:", int(imposibles.sum()))

df_limpio.loc[imposibles, "valor_perdido_cop"] = np.nan

# Se rellena con la mediana del mismo tipo de bien, no con la mediana general
df_limpio["valor_perdido_cop"] = df_limpio["valor_perdido_cop"].fillna(
    df_limpio.groupby("bien_hurtado")["valor_perdido_cop"].transform("median")
)

print("Media del valor perdido:", round(df_limpio["valor_perdido_cop"].mean(), 0))
print("Mediana del valor perdido:", round(df_limpio["valor_perdido_cop"].median(), 0))
print("Máximo después de la corrección:", round(df_limpio["valor_perdido_cop"].max(), 0))

**Decisión:** convierto los 5 valores imposibles en faltantes y los reemplazo por la **mediana del mismo tipo de bien** (la mediana de los celulares para un celular, la de los bolsos para un bolso, etc.). Uso la mediana del grupo y no la general porque una moto y un celular no valen lo mismo: la mediana general le asignaría a un celular un valor de moto.

**Costo:** se corrigen **5 valores** y no se pierde ninguna fila. El efecto se nota en la media, que baja de ≈ 2,79 a ≈ 2,62 millones, mientras la mediana casi no se mueve (1,45 millones). Eso confirma que los 120 millones estaban arrastrando el promedio.

In [ ]:
faltantes_iluminacion = int(df_limpio["iluminacion_zona"].isna().sum())
moda_iluminacion = df_limpio["iluminacion_zona"].mode()[0]

df_limpio["iluminacion_zona"] = df_limpio["iluminacion_zona"].fillna(moda_iluminacion)

print("Moda de iluminacion_zona:", moda_iluminacion)
print("Valores imputados:", faltantes_iluminacion)
print("Faltantes que quedan:", int(df_limpio["iluminacion_zona"].isna().sum()))
print()
print(df_limpio["iluminacion_zona"].value_counts(normalize=True).mul(100).round(1))

**Decisión:** `iluminacion_zona` es una variable **categórica** con solo tres valores (Buena, Regular, Deficiente) y un 4,9 % de faltantes. Para una categórica la estrategia adecuada es la **moda**, que es "Buena" (53,4 % de los casos). No uso la media porque no tiene sentido promediar palabras, y no uso la mediana por grupo porque el porcentaje perdido es bajo y no cambia las conclusiones.

**Costo:** se imputan **219 valores** (4,9 % de la columna) y no se pierde ninguna fila.

In [ ]:
print("Mediana de edad por tipo de incidente:")
print(df_limpio.groupby("tipo_incidente")["edad_victima"].agg(["count", "median"]))
print()
print("Mediana de edad por sexo:")
print(df_limpio.groupby("sexo_victima")["edad_victima"].agg(["count", "median"]))

faltantes_edad = int(df_limpio["edad_victima"].isna().sum())
mediana_edad = df_limpio["edad_victima"].median()
df_limpio["edad_victima"] = df_limpio["edad_victima"].fillna(mediana_edad)

print()
print("Mediana global usada:", mediana_edad, "| Valores imputados:", faltantes_edad)

**Decisión:** uso la **mediana global** (31 años) para `edad_victima`. Antes comprobé que la mediana es la misma (30 o 31 años) en todos los tipos de incidente y en los dos sexos, así que no hay un grupo al que le corresponda un valor distinto: imputar por grupo daría prácticamente el mismo resultado. Uso la mediana y no la media porque la mediana no se deja arrastrar por las edades extremas.

**Costo:** se imputan **392 valores** (8,8 %) y no se pierde ninguna fila.

In [ ]:
print("Mediana del tiempo de respuesta por comuna (minutos):")
print(df_limpio.groupby("comuna")["tiempo_respuesta_min"].agg(["count", "median"]).sort_values("median"))

faltantes_tiempo = int(df_limpio["tiempo_respuesta_min"].isna().sum())
df_limpio["tiempo_respuesta_min"] = df_limpio["tiempo_respuesta_min"].fillna(
    df_limpio.groupby("comuna")["tiempo_respuesta_min"].transform("median")
)

print()
print("Valores imputados:", faltantes_tiempo)
print("Faltantes que quedan:", int(df_limpio["tiempo_respuesta_min"].isna().sum()))

**Decisión:** aquí sí uso la **mediana por comuna**. La tabla muestra que el tiempo de respuesta no es igual en toda la ciudad: en Doce de Octubre la mediana es 34 minutos y en Buenos Aires es 17. Si imputara con una mediana global (20 minutos) le pondría a una comuna lenta un tiempo que no le corresponde y borraría justamente la diferencia que quiero analizar.

**Costo:** se imputan **548 valores** (12,2 %) y no se pierde ninguna fila.

In [ ]:
resumen_limpieza = pd.DataFrame([
    {"decision": "Eliminar filas duplicadas", "filas_antes": 4505, "filas_despues": 4480, "valores_afectados": 25},
    {"decision": "Unificar comunas (52 -> 16)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": comunas_corregidas},
    {"decision": "Eliminar columna testigos", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_testigos},
    {"decision": "Corregir valores imposibles", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": 5},
    {"decision": "Imputar iluminacion_zona (moda)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_iluminacion},
    {"decision": "Imputar edad_victima (mediana global)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_edad},
    {"decision": "Imputar tiempo_respuesta_min (mediana por comuna)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_tiempo},
])
resumen_limpieza

In [ ]:
print("Forma final:", df_limpio.shape)
print()
print("Faltantes que quedan por columna:")
print(df_limpio.isna().sum()[df_limpio.isna().sum() > 0])
print()
print("Filas sin bien_hurtado:", int(df_limpio["bien_hurtado"].isna().sum()))
print("Filas sin valor_perdido_cop:", int(df_limpio["valor_perdido_cop"].isna().sum()))

Al final quedan **4.480 filas y 18 columnas**, y los únicos faltantes que sobreviven son los de `bien_hurtado` y `valor_perdido_cop` (1.477 cada uno). Esos son exactamente los **"no aplica"** de riñas, lesiones y violencia intrafamiliar, y se dejan así a propósito: un vacío que significa "no hubo bien hurtado" es información, no un error.

Las columnas que sí usé en el análisis (`comuna`, `iluminacion_zona`, `edad_victima`, `tiempo_respuesta_min`) quedaron completas.

## 4. Preguntas de análisis

Con los datos limpios respondo cinco preguntas que sirven para tomar decisiones de seguridad en la ciudad. Todas usan **filtrado booleano**, **`loc`**, **`sort_values`** y **`groupby` con agregaciones**, e incluyo `count` para saber con cuántos datos se calculó cada promedio.

### Pregunta 1. ¿Qué tipo de incidente es el más frecuente y cuánto pesa en el total?

Sirve para saber dónde se concentra el esfuerzo de prevención. Uso `value_counts()` y ordeno con `sort_values()`.

In [ ]:
conteo_tipos = df_limpio["tipo_incidente"].value_counts().to_frame("incidentes")
conteo_tipos["porcentaje"] = (conteo_tipos["incidentes"] / len(df_limpio) * 100).round(1)
conteo_tipos.sort_values("incidentes", ascending=False)

**Respuesta:** el hurto a persona es el más frecuente con **1.359 casos (30,3 %)**, seguido del hurto de celular con **1.245 (27,8 %)**. Si se suman las tres modalidades de hurto (persona, celular y moto) se llega a **3.003 casos, el 67,0 % del total**. Los delitos de convivencia (riña, violencia intrafamiliar y lesiones) son el 33,0 % restante.

**Implicación:** la prevención del hurto, y en particular del hurto a persona y de celular, es la prioridad si el objetivo es bajar el número total de incidentes.

### Pregunta 2. ¿En qué comunas se concentran los hurtos y cuánto vale lo que se pierde?

Uso `loc` para quedarme solo con los hurtos, filtro booleano para los casos de mayor valor y `groupby` con `count` y `median`.

In [ ]:
hurtos = df_limpio.loc[df_limpio["tipo_incidente"].str.startswith("Hurto"), :]
print("Hurtos en la base:", len(hurtos))

hurtos_por_comuna = hurtos.groupby("comuna").agg(
    hurtos=("id_incidente", "count"),
    valor_mediano=("valor_perdido_cop", "median"),
)
hurtos_por_comuna["porcentaje"] = (hurtos_por_comuna["hurtos"] / len(hurtos) * 100).round(1)
hurtos_por_comuna.sort_values("hurtos", ascending=False).round(0)

In [ ]:
hurtos.loc[hurtos["valor_perdido_cop"] > 15_000_000,
           ["id_incidente", "comuna", "bien_hurtado", "valor_perdido_cop"]
          ].sort_values("valor_perdido_cop", ascending=False).head(8)

**Respuesta:** los hurtos se concentran en tres comunas: **La Candelaria (611, 20,0 %)**, **El Poblado (384, 13,0 %)** y **Laureles Estadio (329, 11,0 %)**. Entre las tres reúnen **1.324 hurtos, el 44,1 %** de todos los hurtos de la ciudad. En las comunas con menos hurtos, como Santa Cruz (68) o Popular (84), el valor mediano perdido es parecido al de las comunas grandes (alrededor de 1,2 a 1,6 millones), así que la diferencia está en la **cantidad** de casos, no en el monto.

Los casos de mayor valor son casi todos **motocicletas** (entre 20 y 32 millones), y La Candelaria aparece varias veces en esa lista.

**Implicación:** una campaña concentrada en tres comunas cubriría casi la mitad de los hurtos. Para las motos, que son lo más costoso, conviene reforzar parqueaderos y controles en La Candelaria.

### Pregunta 3. ¿Qué comunas tienen el tiempo de respuesta más lento?

El tiempo de respuesta es lo que más se puede mejorar con logística. Agrupo por comuna y comparo la mediana, que no se deja arrastrar por las demoras extremas.

In [ ]:
tiempo_por_comuna = df_limpio.groupby("comuna").agg(
    incidentes=("id_incidente", "count"),
    tiempo_mediano=("tiempo_respuesta_min", "median"),
)
tiempo_por_comuna.sort_values("tiempo_mediano", ascending=False).round(1)

**Respuesta:** la mediana del tiempo de respuesta va de **17 minutos en Buenos Aires a 34 minutos en Doce de Octubre**, es decir, el doble. Las comunas más lentas son **Doce de Octubre (34), Santa Cruz (33), Popular (32,5) y Villa Hermosa (32)**; las más rápidas son Buenos Aires (17), La Candelaria, Aranjuez y Robledo (18).

Las comunas lentas no son las que tienen más incidentes (La Candelaria, la que más tiene, responde en 18 minutos). Eso sugiere que el problema no es el volumen de casos sino la **ubicación o la capacidad de respuesta** en la periferia.

**Implicación:** reubicar o reforzar unidades en el nororiente y el occidente (Doce de Octubre, Santa Cruz, Popular, Villa Hermosa) reduciría la brecha de respuesta.

### Pregunta 4. ¿Qué proporción de los incidentes se denuncia, según el tipo?

La denuncia es la puerta de entrada a la investigación. Comparo el porcentaje con `groupby`, `count` y `sum` sobre una columna booleana.

In [ ]:
df_limpio["denuncio"] = df_limpio["denunciado"] == "Sí"

denuncias_por_tipo = df_limpio.groupby("tipo_incidente").agg(
    incidentes=("id_incidente", "count"),
    denuncias=("denuncio", "sum"),
)
denuncias_por_tipo["porcentaje_denuncia"] = (
    denuncias_por_tipo["denuncias"] / denuncias_por_tipo["incidentes"] * 100
).round(1)
denuncias_por_tipo.sort_values("porcentaje_denuncia", ascending=False)

**Respuesta:** en total se denuncia el **48,1 %** de los incidentes. El porcentaje más alto es el del **hurto de moto (65,2 % de 399 casos)** y el más bajo el de la **riña (36,6 % de 609 casos)**. La violencia intrafamiliar (52,7 %) y el hurto de celular (51,0 %) quedan por encima del promedio; el hurto a persona (45,6 %) y las lesiones personales (42,4 %) por debajo.

Tiene sentido que la moto se denuncie más: es un bien caro y muchas veces asegurado, así que la denuncia sirve para el seguro. La riña, en cambio, se resuelve en el momento y se denuncia menos.

**Implicación:** las cifras de riñas y lesiones están más subestimadas que las de hurto. Si se quiere medir la violencia interpersonal, hay que acompañar la denuncia con campañas o con registros de urgencias.

### Pregunta 5. ¿Cómo se relacionan la iluminación de la zona y el tipo de incidente?

Es una pregunta de prevención situacional: si los delitos de convivencia ocurren más en zonas con mala iluminación, la intervención es distinta a la del hurto. Uso una tabla cruzada con porcentajes por fila.

In [ ]:
iluminacion_por_tipo = pd.crosstab(
    df_limpio["tipo_incidente"],
    df_limpio["iluminacion_zona"],
    normalize="index",
).mul(100).round(1)
iluminacion_por_tipo.sort_values("Deficiente", ascending=False)

In [ ]:
df_limpio.groupby("iluminacion_zona").agg(
    incidentes=("id_incidente", "count"),
    camaras_promedio=("camaras_cercanas", "mean"),
    tiempo_mediano=("tiempo_respuesta_min", "median"),
).round(2)

**Respuesta:** los delitos de convivencia ocurren en zonas con **peor iluminación** que los hurtos. El **32,0 % de las violencias intrafamiliares** y el **31,0 % de las riñas** están en zonas de iluminación deficiente, frente al **13,9 % del hurto a persona**. Los hurtos, en cambio, se concentran en zonas de iluminación buena (58,9 % del hurto a persona).

La segunda tabla muestra que la iluminación viene acompañada de otros factores: las zonas de iluminación **deficiente tienen en promedio 1,91 cámaras cercanas y una respuesta mediana de 26 minutos**, mientras las de iluminación **buena tienen 2,75 cámaras y 19 minutos**.

**Implicación:** la iluminación no es la única causa, pero las zonas deficientes acumulan tres desventajas a la vez (menos luz, menos cámaras y respuesta más lenta). Ahí es donde una intervención integral tendría más efecto.

## 5. Conclusiones parciales

1. **La base es utilizable después de la limpieza.** De 4.505 filas quedaron **4.480** (solo se eliminaron 25 duplicados, 0,6 %), con 18 columnas. Las columnas usadas en el análisis quedaron completas.

2. **El problema principal es el hurto.** El 67,0 % de los incidentes son hurtos, y el hurto a persona y el de celular son el 58,1 % del total.

3. **La concentración geográfica es fuerte.** La Candelaria, El Poblado y Laureles Estadio reúnen el 44,1 % de los hurtos.

4. **La respuesta no es igual en toda la ciudad.** La mediana va de 17 a 34 minutos según la comuna, y las comunas lentas no son las de más casos.

5. **La denuncia depende del tipo de delito.** El hurto de moto se denuncia en el 65,2 % de los casos y la riña en el 36,6 %, así que las cifras de violencia interpersonal están más subestimadas.

6. **Iluminación, cámaras y tiempo de respuesta van juntos.** Las zonas con iluminación deficiente tienen menos cámaras (1,91 frente a 2,75) y respuesta más lenta (26 frente a 19 minutos).

